# SmartLine Kaggle training

Primary cloud training workflow for SmartLine AI. Use a GPU accelerator.
This notebook installs the pinned stack, checks the labeled dataset, trains YOLO26, validates, and exports ONNX plus OpenVINO.
It does not embed credentials. If the dataset or labels are missing, it stops without printing accuracy, mAP, precision, or recall.

Add two Kaggle inputs: this repository and a labeled YOLO dataset produced by `training/dataset_tools.py export-yolo`.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit('CUDA is not available. Select a Kaggle GPU accelerator before training.')
name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU: {name}')
print(f'VRAM_GB: {vram_gb:.2f}')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
from pathlib import Path
import subprocess
import sys

requirements = next(Path('/kaggle/input').rglob('training/requirements.txt'), None)
if requirements is None:
    raise SystemExit('Add the SmartLine repository as a Kaggle input so training/requirements.txt can be installed.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])
print(f'Installed pins from {requirements}')

In [ ]:
from pathlib import Path
import sys

repo = next(path.parents[1] for path in Path('/kaggle/input').rglob('training/train.py'))
sys.path.insert(0, str(repo / 'training'))
import dataset_tools
import train

candidates = [path.parent for path in Path('/kaggle/input').rglob('data.yaml')]
if not candidates:
    raise SystemExit('No data.yaml was found. Upload the labeled YOLO export. No metric was produced.')
dataset = candidates[0]
ready, reason = train.dataset_status(dataset, 'detect')
print(dataset)
print(reason)
if not ready:
    raise SystemExit('Dataset is not trainable. No metric was produced.')
labels = dataset_tools.validate_label_files(dataset, class_count=3)
print(labels)
if labels['problems']:
    raise SystemExit('Label validation failed. No metric was produced.')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
train_result = model.train(
    data=str(dataset / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    patience=20,
    seed=42,
    project='/kaggle/working/runs',
    name='demobox-detect',
    exist_ok=False,
)
metrics = model.val(data=str(dataset / 'data.yaml'), device=0)
print(metrics.results_dict)
print('Confusion matrix and results.csv are in the run directory written by Ultralytics.')

In [ ]:
from pathlib import Path
import shutil

best = Path('/kaggle/working/runs/demobox-detect/weights/best.pt')
if not best.is_file():
    raise SystemExit(f'Trained checkpoint is missing: {best}')
from export import execute as export_checkpoint
export_checkpoint(best)
bundle = Path('/kaggle/working/smartline-model-bundle')
bundle.mkdir(exist_ok=True)
shutil.copy2(best, bundle / 'best.pt')
print('onnx', best.with_suffix('.onnx'))
print('openvino', str(best.with_suffix('')) + '_openvino_model')
print('Download /kaggle/working/runs and the exported files. Do not register the model until the real results.csv has been reviewed.')